# Projekat 2

Dataset se može pronaći na sledećem linku [Indian E-Commerce Customer Behavior & Purchase](https://www.kaggle.com/datasets/kundanbedmutha/indian-e-commerce-customer-behavior-and-purchase).

## Imports

In [17]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.cluster.hierarchy as sch
import mpl_toolkits.mplot3d.axes3d as p3
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn import metrics, mixture, cluster, datasets
from sklearn.mixture import GaussianMixture
from sklearn.datasets import make_moons
from sklearn.cluster import AgglomerativeClustering, KMeans, DBSCAN
from sklearn.neighbors import kneighbors_graph
from itertools import cycle, islice
import time
from math import ceil
from scipy.stats import kurtosis, skew, iqr, boxcox, hmean
import copy

## Učitavanje dataset-a

In [3]:
dataset=pd.read_csv('./Ecommerce.csv')
display(dataset.columns)

Index(['customer_id', 'session_id', 'visit_date', 'device_type', 'user_type',
       'marketing_channel', 'product_id', 'product_category', 'unit_price',
       'quantity', 'discount_percent', 'discount_amount', 'revenue',
       'pages_viewed', 'time_on_site_sec', 'added_to_cart', 'purchased',
       'cart_abandoned', 'rating', 'review_text', 'review_helpful_votes',
       'payment_method', 'visit_day', 'visit_month', 'visit_weekday',
       'visit_season', 'session_duration_bucket', 'revenue_normalized',
       'location'],
      dtype='object')

## Opis atributa dataset-a


| Kolona                  | Opis                                                                 |
|--------------------------|----------------------------------------------------------------------|
| customer_id              | Jedinstveni identifikator kupca                                      |
| session_id               | Jedinstveni identifikator posete/sesije                              |
| visit_date               | Datum posete u formatu DD-MM-YYYY                                    |
| device_type              | Uređaj korišćen (0=Desktop, 1=Mobilni, 2=Tablet)                     |
| user_type                | Tip korisnika (0=Novi, 1=Stalni)                                  |
| marketing_channel        | Kanal akvizicije (kodirano 0–5)                                      |
| product_id               | Identifikator proizvoda                                              |
| product_category         | Kodirana kategorija proizvoda                                        |
| unit_price               | Cena proizvoda pre popusta                                           |
| quantity                 | Kupljena količina proizvoda                                          |
| discount_percent         | Procenat popusta primenjen na proizvod                               |
| discount_amount          | Iznos popusta u valuti                                               |
| revenue                  | Ukupan prihod od transakcije                                         |
| pages_viewed             | Broj pregledanih stranica tokom sesije                               |
| time_on_site_sec         | Vreme provedeno na sajtu (u sekundama)                               |
| added_to_cart            | Da li je proizvod dodat u korpu (0/1)                                |
| purchased                | Da li je kupovina završena (0/1)                                     |
| cart_abandoned           | Da li je korpa napuštena (0/1)                                       |
| rating                   | Ocena proizvoda/usluge koju je dao kupac (1–5)                      |
| review_text              | Tekstualna recenzija kupca                                           |
| review_helpful_votes     | Broj glasova da je recenzija bila korisna                            |
| payment_method           | Način plaćanja (kodirane kategorije)                                 |
| visit_day                | Dan u mesecu kada je poseta obavljena                                |
| visit_month              | Mesec posete                                                         |
| visit_weekday            | Dan u nedelji kada je poseta obavljena                               |
| visit_season             | Sezona posete (Zima/Proleće/Leto/Jesen)                              |
| session_duration_bucket  | Trajanje sesije grupisano u kategorije                               |
| revenue_normalized       | Normalizovan prihod radi poređenja                                   |
| location                 | Lokacija kupca (grad/region)                                         |


In [4]:
display(dataset.head(5))

,customer_id,session_id,visit_date,device_type,user_type,marketing_channel,product_id,product_category,unit_price,quantity,...,review_text,review_helpful_votes,payment_method,visit_day,visit_month,visit_weekday,visit_season,session_duration_bucket,revenue_normalized,location
0,1803,0,28-11-2024,2,1,2,894,6,651.57,1,...,1,0,1,28,11,3,0,Long,0.000000,209
1,7964,1,25-09-2024,2,0,4,844,2,945.27,4,...,1,0,2,25,9,2,0,Long,0.000000,213
2,6890,2,31-05-2024,1,1,0,865,0,400.44,4,...,1,0,2,31,5,4,1,Short,0.000000,10
3,4949,3,30-01-2024,1,0,2,851,3,1268.54,2,...,10,4,1,30,1,1,3,Very Long,0.305504,46
4,4896,4,25-02-2024,1,1,5,794,3,880.81,3,...,1,0,1,25,2,6,3,Very Short,0.000000,118


In [5]:
display(dataset.shape)

(25000, 29)

In [6]:
display(dataset.dtypes)
display(dataset.dtypes.value_counts())

customer_id                  int64
session_id                   int64
visit_date                  object
device_type                  int64
user_type                    int64
marketing_channel            int64
product_id                   int64
product_category             int64
unit_price                 float64
quantity                     int64
discount_percent             int64
discount_amount            float64
revenue                    float64
pages_viewed                 int64
time_on_site_sec             int64
added_to_cart                int64
purchased                    int64
cart_abandoned               int64
rating                       int64
review_text                  int64
review_helpful_votes         int64
payment_method               int64
visit_day                    int64
visit_month                  int64
visit_weekday                int64
visit_season                 int64
session_duration_bucket     object
revenue_normalized         float64
location            

int64      23
float64     4
object      2
Name: count, dtype: int64

In [7]:
def column_values(dataset:pd.DataFrame):
    for col in dataset.columns:
        temp=pd.unique(dataset[col])
        if dataset[col].dtype==object:
            print(f'Column: {col}, has {len(temp)} unique values. {temp}')
        else:
            print(f'Column: {col}, has {len(temp)} unique values, {temp.min()}-{temp.max()}')
            
display(column_values(dataset))

Column: customer_id, has 8442 unique values, 1000-9998
Column: session_id, has 25000 unique values, 0-24999
Column: visit_date, has 365 unique values. ['28-11-2024' '25-09-2024' '31-05-2024' '30-01-2024' '25-02-2024'
 '29-03-2024' '28-05-2024' '19-11-2024' '13-02-2024' '29-07-2024'
 '06-05-2024' '18-01-2024' '14-08-2024' '14-09-2024' '16-12-2024'
 '03-07-2024' '28-04-2024' '07-11-2024' '31-03-2024' '06-09-2024'
 '10-12-2024' '02-01-2024' '03-12-2024' '16-03-2024' '24-06-2024'
 '04-04-2024' '22-02-2024' '13-09-2024' '04-07-2024' '25-06-2024'
 '29-09-2024' '18-12-2024' '06-07-2024' '26-02-2024' '29-04-2024'
 '30-06-2024' '07-02-2024' '24-03-2024' '14-03-2024' '21-12-2024'
 '26-05-2024' '16-11-2024' '20-07-2024' '08-12-2024' '09-09-2024'
 '28-03-2024' '12-04-2024' '31-08-2024' '01-01-2024' '28-01-2024'
 '17-09-2024' '10-06-2024' '05-01-2024' '27-09-2024' '27-08-2024'
 '19-09-2024' '29-02-2024' '15-11-2024' '06-03-2024' '27-07-2024'
 '23-07-2024' '20-03-2024' '17-05-2024' '20-01-2024' '21-

None

## Provera postojanja duplikata i nedostajućih vrednosti

In [8]:
display(dataset[dataset.duplicated()])

,customer_id,session_id,visit_date,device_type,user_type,marketing_channel,product_id,product_category,unit_price,quantity,...,review_text,review_helpful_votes,payment_method,visit_day,visit_month,visit_weekday,visit_season,session_duration_bucket,revenue_normalized,location


In [9]:
print(dataset.isna().mean() * 100)

customer_id                0.0
session_id                 0.0
visit_date                 0.0
device_type                0.0
user_type                  0.0
marketing_channel          0.0
product_id                 0.0
product_category           0.0
unit_price                 0.0
quantity                   0.0
discount_percent           0.0
discount_amount            0.0
revenue                    0.0
pages_viewed               0.0
time_on_site_sec           0.0
added_to_cart              0.0
purchased                  0.0
cart_abandoned             0.0
rating                     0.0
review_text                0.0
review_helpful_votes       0.0
payment_method             0.0
visit_day                  0.0
visit_month                0.0
visit_weekday              0.0
visit_season               0.0
session_duration_bucket    0.0
revenue_normalized         0.0
location                   0.0
dtype: float64


## Histogrami dataset-a

In [10]:
def draw_histograms(dataset, columns, fsize=(40,200), color='#6ac2e2'):
    num_cols=2
    num_rows=(len(columns)+1)//2
    fig,axes=plt.subplots(nrows=num_rows, ncols=num_cols, figsize=fsize)
    for ax,column in zip(axes.flatten(), columns):
        if(dataset[column].dtype==object):
            sns.histplot(dataset[column], ax=ax, color=color)
            counts = dataset[column].value_counts().sort_index()
            bars = ax.patches
            for bar, label in zip(bars, counts.index):
                height = bar.get_height()
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    height / 2,
                    label,
                    ha='center', va='center',
                    fontsize=20, color='white', weight='bold',
                    rotation=90
                )
            ax.set_xticks([])
 
        else:
            sns.histplot(dataset[column], ax=ax, color=color, kde=False)
 
        ax.set_title(column, fontsize=43)
        ax.tick_params(axis='both', which='major', labelsize=35)
        ax.tick_params(axis='both', which='minor', labelsize=35)
        ax.set_xlabel('')
    fig.tight_layout(rect=[0,0.03,1.0,0.7], h_pad=10)
    
draw_histograms(dataset, dataset.columns)

## Ordinalno enkodiranje

In [13]:
pd.set_option('future.no_silent_downcasting', True)
dataset_encoded=dataset.copy(deep=True)
session_duration_mapper = {
        "Very Short": 1,
        "Short": 2,
        "Long": 3,
        "Very Long": 4,
    }
dataset_encoded["session_duration_bucket"] = dataset_encoded["session_duration_bucket"].replace(session_duration_mapper).astype(int)

display(dataset_encoded.dtypes.value_counts())

int64      24
float64     4
object      1
Name: count, dtype: int64

# RESITI DATUM

## Deskriptivna analiza

In [20]:
def series_analysis(column: pd.Series, column_name):
    res_disct = dict()
    res_disct["name"] = column_name

    sorted_column = column.sort_values()

    res_disct["mean"] = column.mean()
    res_disct["median"] = column.median()
    res_disct["mode"] = column.mode().values.tolist()
    
    if (column > 0).all():
        res_disct["geometric_mean"] = np.exp(np.mean(np.log(column)))
    else:
        res_disct["geometric_mean"] = None
        
    res_disct["harmonic_mean"] = hmean(column[column > 0])
    res_disct["max"] = float(column.max())
    res_disct["min"] = float(column.min())
    res_disct["range"] = float(column.max() - column.min())
    res_disct["std"] = column.std(ddof=1)
    res_disct["variance"] = column.var(ddof=1)
    res_disct["skewness"] = skew(column)
    res_disct["kurtosis"] = kurtosis(column)
    res_disct["iqr"] = iqr(sorted_column.values)

    res_disct["sorted"] = sorted_column.values.tolist()

    return res_disct


def descryptive_analysis(dataset: pd.DataFrame):

    column_dict = {}
    for col in dataset.columns:
        temp = series_analysis(dataset[col], col)
        temp2 = copy.deepcopy(temp)
        del temp2["name"]
        column_dict[col] = temp2

    return column_dict


dataset_no_date = dataset_encoded.drop(columns=["visit_date"])
column_dict = descryptive_analysis(dataset_no_date)
mc=pd.get_option('display.max_columns')
pd.set_option('display.max_columns',None)
display(pd.DataFrame(column_dict))
pd.set_option('display.max_columns',mc)

,customer_id,session_id,device_type,user_type,marketing_channel,product_id,product_category,unit_price,quantity,discount_percent,discount_amount,revenue,pages_viewed,time_on_site_sec,added_to_cart,purchased,cart_abandoned,rating,review_text,review_helpful_votes,payment_method,visit_day,visit_month,visit_weekday,visit_season,session_duration_bucket,revenue_normalized,location
mean,5479.9306,12499.5,0.70504,0.55128,2.51404,449.107,3.49524,782.31901,2.48904,8.9988,174.997669,404.646762,12.53584,903.26292,0.64468,0.22464,0.42004,3.948,1.67488,5.5208,2.48316,15.71452,6.51052,2.9808,1.5044,2.49856,0.05129,111.70692
median,5482.0,12499.5,1.0,1.0,2.0,448.0,4.0,691.725,2.0,10.0,65.815,0.0,13.0,903.0,1.0,0.0,0.0,4.0,1.0,0.0,2.0,16.0,7.0,3.0,2.0,2.0,0.0,111.0
mode,[7811],"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",[1],[1],[2],[499],[2],"[343.45, 456.76, 559.83, 590.1, 974.39]",[2],[0],[0.0],[0.0],[2],[141],[1],[0],[0],[4],[1],[0],[1],[20],[8],[4],[3],[1],[0.0],[46]
geometric_mean,4732.896424,None,None,None,None,None,None,627.430688,2.204142,None,None,None,9.825511,677.80248,None,None,None,3.8918,None,None,None,12.187234,5.291747,None,None,2.211667,None,None
harmonic_mean,3893.325745,2335.51986,1.090679,1.0,2.187251,122.754827,2.710801,466.801233,1.913473,11.194108,98.74316,800.832625,6.355329,339.783469,1.0,1.0,1.0,3.792562,1.137745,11.132255,2.173329,7.548805,3.857512,2.442104,1.637822,1.918257,0.101508,37.830078
max,9998.0,24999.0,2.0,1.0,5.0,898.0,7.0,1999.83,4.0,30.0,2388.26,7889.36,24.0,1799.0,1.0,1.0,1.0,5.0,10.0,49.0,5.0,31.0,12.0,6.0,3.0,4.0,1.0,224.0
min,1000.0,0.0,0.0,0.0,0.0,0.0,0.0,50.05,1.0,0.0,0.0,0.0,1.0,10.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
range,8998.0,24999.0,2.0,1.0,5.0,898.0,7.0,1949.78,3.0,30.0,2388.26,7889.36,23.0,1789.0,1.0,1.0,1.0,4.0,10.0,49.0,5.0,30.0,11.0,6.0,3.0,3.0,1.0,224.0
std,2593.544596,7217.022701,0.639543,0.497373,1.704934,259.513787,2.285053,476.612168,1.114563,9.26364,269.005411,1022.391774,6.929762,518.676202,0.47862,0.417353,0.493575,0.54047,1.98673,12.33629,1.709501,8.796338,3.457925,1.992554,1.117475,1.118413,0.129591,65.07079
variance,6726473.570326,52085416.666667,0.409015,0.24738,2.906799,67347.405767,5.221466,227159.158711,1.24225,85.815031,72363.911294,1045284.938627,48.021596,269025.002073,0.229077,0.174184,0.243616,0.292108,3.947095,152.184055,2.922393,77.375556,11.957248,3.97027,1.248751,1.250848,0.016794,4234.207752
